# SQuaRE — interactive report (frozen concepts)

Sliders bind only to keys in `square.mc.overrides.PARAMETER_LAYERS` that exist as `parameter_entry` values in the loaded modality/QEC YAML. Rebuild uses `build_scenario_report`; the table below is `square.plotting.extract_report_plot_frame` (failure proxy, magic flags, schedule ratio).

**Setup:** from the `SQUARE/` directory, `pip install -e ".[interactive]"`, then open this notebook in Jupyter / VS Code.

In [ ]:
from pathlib import Path

from IPython.display import Image, display
import ipywidgets as W
from square.loader import find_square_root, load_scenario_bundle
from square.mc.overrides import PARAMETER_LAYERS, apply_numeric_overrides
from square.plotting import extract_report_plot_frame, write_report_semantics_png
from square.report import build_scenario_report
from square.yaml_assumption import is_parameter_entry

ROOT = find_square_root(Path.cwd().resolve())
SCENARIO = ROOT / "Configs" / "oratomic_gold_path.yaml"
bundle0 = load_scenario_bundle(SCENARIO, root=ROOT, require_scenario_under_root=True)

slider_widgets: dict[str, W.FloatSlider] = {}
for key, layer in PARAMETER_LAYERS.items():
    doc = bundle0.modality if layer == "modality" else bundle0.qec
    entry = doc.get(key)
    if not is_parameter_entry(entry):
        continue
    v = float(entry["value"])
    lo, hi = (v * 0.25, v * 4.0) if v != 0 else (-1e-6, 1e-6)
    if lo > hi:
        lo, hi = hi, lo
    slider_widgets[key] = W.FloatSlider(value=v, min=lo, max=hi, step=(hi - lo) / 200, description=key[:18])

d_widget = W.IntText(value=0, description="d override (0=auto)")
ui = W.VBox([W.HTML("<b>PARAMETER_LAYERS</b>"), *slider_widgets.values(), d_widget])
display(ui)

In [ ]:
from IPython.display import Image, display


def rebuild(_=None):
    ov = {k: float(w.value) for k, w in slider_widgets.items()}
    b = apply_numeric_overrides(bundle0, ov) if ov else bundle0
    d = int(d_widget.value) if d_widget.value > 0 else None
    rep = build_scenario_report(b, code_distance_override=d)
    frame = extract_report_plot_frame(rep)
    display(frame)
    for line in rep.get("warnings", [])[:12]:
        print(line)
    if len(rep.get("warnings", [])) > 12:
        print("...")
    outp = Path("_notebook_last_semantics.png")
    write_report_semantics_png(outp, rep)
    display(Image(filename=str(outp)))


go = W.Button(description="Build report", button_style="primary")
go.on_click(rebuild)
display(go)

After clicking **Build report**, the semantics PNG is shown inline above the plot file path (`_notebook_last_semantics.png` in the current working directory).